In [1]:
# Quantitative Volatility Modeling Using ARCH, GARCH & EWMA Models

In [2]:
import numpy as np
import pandas as pd

In [3]:
# Import data from Yahoo Finance
import yfinance as yf
symbol = 'JPM'
df = yf.download(symbol, start = '2015-01-01', end = '2026-01-01')
df = df['Close']
df

[*********************100%***********************]  1 of 1 completed


Ticker,JPM
Date,
2015-01-02,45.838104
2015-01-05,44.415051
2015-01-06,43.263432
2015-01-07,43.329449
2015-01-08,44.297691
...,...
2025-12-24,322.950073
2025-12-26,321.713898
2025-12-29,317.632477


ARCH Model

In [4]:
from arch import arch_model

In [5]:
# Step 1: Get the data from Yahoo Finance
df = yf.download('JPM', start = '2020-01-01', end = '2025-01-01')
df['returns'] = df['Close'].pct_change()*100 # daily % return
returns = df['returns'].dropna()
model = arch_model(returns, vol = 'ARCH', p = 1) # Call the model
results = model.fit(disp = 'off') # Train the model
results.summary()

[*********************100%***********************]  1 of 1 completed


<class 'statsmodels.iolib.summary.Summary'>
"""
                      Constant Mean - ARCH Model Results                      
==============================================================================
Dep. Variable:                returns   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                       ARCH   Log-Likelihood:               -2494.78
Distribution:                  Normal   AIC:                           4995.56
Method:            Maximum Likelihood   BIC:                           5010.96
                                        No. Observations:                 1257
Date:                Thu, Oct 29 2026   Df Residuals:                     1256
Time:                        05:21:47   Df Model:                            1
                                Mean Model                                
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.1558  5.604e-02      2.781  5.416e-03 [4.602e-02,  0.266]
                            Volatility Model                            
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
omega          2.0425      0.239      8.555  1.183e-17 [  1.575,  2.510]
alpha[1]       0.5212  9.643e-02      5.405  6.488e-08 [  0.332,  0.710]
========================================================================

Covariance estimator: robust
"""

In [6]:
# Step 6: Forecast 5 days ahead

forecast = results.forecast(horizon = 5)
predicted_variance = forecast.variance
predicted_volatility = predicted_variance ** 0.5
predicted_volatility

,h.1,h.2,h.3,h.4,h.5
Date,,,,,
2024-12-31,1.42918,1.762698,1.91362,1.987744,2.025301


In [7]:
predicted_volatility = [1.479269, 1.567023, 1.577429, 1.578698, 1.578853]
predicted_avg_vol = sum(predicted_volatility)/len(predicted_volatility)
predicted_avg_vol

1.5562543999999998

In [8]:
# Step 7: Get the data and calculate realized volatility

start_date = pd.to_datetime('2024-12-31')
end_date = pd.to_datetime('2025-01-09') #  buffer for weekends and holida
real_df = yf.download('JPM', start_date, end_date)
real_df['returns'] = real_df['Close'].pct_change()*100
real_df = real_df.dropna()
realized_vol = real_df['returns'].std()*np.sqrt(5) # Predicting the vol 
print(realized_vol)

[*********************100%***********************]  1 of 1 completed

1.691967979273258


In [9]:
print("ARCH Model Predicted Volatility:", predicted_avg_vol)
print("ARCH Model Actual Volatility:", realized_vol)

ARCH Model Predicted Volatility: 1.5562543999999998
ARCH Model Actual Volatility: 1.691967979273258


GARCH Model

In [10]:
from arch import arch_model

In [11]:
# Step 1: Get the data from Yahoo Finance
df = yf.download('JPM', start = '2020-01-01', end = '2026-01-01')
df['returns'] = df['Close'].pct_change()*100
returns = df['returns'].dropna()
model = arch_model(returns, vol = 'GARCH', p = 1, q = 1)
results = model.fit(disp = 'off') 
results.summary()

[*********************100%***********************]  1 of 1 completed


<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                returns   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2891.91
Distribution:                  Normal   AIC:                           5791.82
Method:            Maximum Likelihood   BIC:                           5813.09
                                        No. Observations:                 1507
Date:                Thu, Oct 29 2026   Df Residuals:                     1506
Time:                        05:21:49   Df Model:                            1
                                Mean Model                                
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.1455  4.289e-02      3.392  6.929e-04 [6.143e-02,  0.230]
                             Volatility Model                             
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.3574      0.166      2.151  3.146e-02 [3.178e-02,  0.683]
alpha[1]       0.1709  6.390e-02      2.674  7.487e-03 [4.565e-02,  0.296]
beta[1]        0.7193  9.869e-02      7.289  3.126e-13   [  0.526,  0.913]
==========================================================================

Covariance estimator: robust
"""

In [12]:
# Step 6: Forecast 5 days ahead
forecast = results.forecast(horizon = 5)
predicted_variance = forecast.variance
predicted_volatility = predicted_variance ** 0.5
predicted_volatility

,h.1,h.2,h.3,h.4,h.5
Date,,,,,
2025-12-31,1.321737,1.382946,1.435241,1.48024,1.519178


In [13]:
predicted_volatility = [1.623483, 1.622252, 1.621032, 1.619824, 1.618626]
predicted_avg_vol = sum(predicted_volatility)/len(predicted_volatility)
predicted_avg_vol

1.6210434

In [14]:
start_date = pd.to_datetime('2024-12-31')
end_date = pd.to_datetime('2026-07-31')
real_df = yf.download('JPM', start_date, end_date)
real_df['returns'] = real_df['Close'].pct_change()*100
real_df = real_df.dropna()
realized_vol = real_df['returns'].std()*np.sqrt(5)
print(realized_vol)

[*********************100%***********************]  1 of 1 completed

3.4639002980152007


In [15]:
print("GARCH Model Predicted Volatility:", predicted_avg_vol)
print("GARCH Model Actual Volatility:", realized_vol)

GARCH Model Predicted Volatility: 1.6210434
GARCH Model Actual Volatility: 3.4639002980152007


EWMA Model

In [16]:
# Step 1: Download the data
df = yf.download('JPM', start = '2022-01-01', end = '2025-01-01')
df['returns'] = df['Close'].pct_change()
df = df.dropna()
lamda = 0.94
ewma_var = []
var_t = df['returns'].var()
for ret in df['returns']:
    variance_tplus1 = lamda*var_t + (1-lamda)* (ret**2)
    ewma_var.append(variance_tplus1)
df['ewma_vol'] = np.sqrt(ewma_var)
latest_daily_vol = df['ewma_vol'].iloc[-1]
print(latest_daily_vol)

[*********************100%***********************]  1 of 1 completed

0.015268655215521966


In [17]:
start_date = pd.to_datetime('2024-12-31')
end_date = pd.to_datetime('2026-07-31') #  buffer for weekends and holida
real_df = yf.download('JPM', start_date, end_date)
real_df['returns'] = real_df['Close'].pct_change()*100
real_df = real_df.dropna()
realized_vol = real_df['returns']
realized_vol

[*********************100%***********************]  1 of 1 completed


Date
2025-01-02    0.120968
2025-01-03    1.366671
2025-01-06   -0.487542
2025-01-07    0.963251
2025-01-08   -0.016439
                ...   
2026-07-24    0.945988
2026-07-27    0.846522
2026-07-28    0.311623
2026-07-29   -3.526356
2026-07-30    1.781219
Name: returns, Length: 394, dtype: float64